# JNPA UC-III — Congestion Forecaster: train on Google Colab (free CPU)

Trains the **GraphSAGE + LSTM** congestion-onset forecaster entirely in Colab's
compute, then downloads the two small artifacts so your device stores almost nothing:

- `congestion_gnn_lstm.pt`  (~270 KB)
- `metrics.json`            (~1 KB)

The model is tiny — **free CPU runtime is enough, no GPU needed** (Runtime ->
Change runtime type -> CPU is fine). Training uses 14+ days of deterministic
synthetic commute history, so **no database or infra is required**.

## What this fixes
The committed checkpoint reports **F1 = 0.8411**, just under the **0.85** bid
target. The trainer has a built-in retry loop that up-adjusts the positive-class
weight until the F1/precision/recall targets are met. We raise `MAX_RETRIES` and
`HISTORY_DAYS` below to give it room to push F1 over 0.85.

## How to use
1. Open this notebook in Colab (File -> Upload notebook, or push the repo and
   open from GitHub).
2. **Set `REPO_URL`** in the first code cell to your repo's git URL (or upload
   the repo as a zip — see the alternative cell).
3. Runtime -> Run all.
4. The last cell downloads `congestion-artifacts.zip`. Unzip it into
   `ai/congestion/artifacts/` in your local repo, overwriting the two files.

## 1. Get the repo into Colab

**Option A — clone from git** (set `REPO_URL`). If the repo is private, use a
token URL, e.g. `https://<TOKEN>@github.com/you/jnpa-uc3-poc.git`.

In [ ]:
REPO_URL = ""  # <-- set me, e.g. https://github.com/you/jnpa-uc3-poc.git
REPO_DIR = "jnpa-uc3-poc"

import os, subprocess, sys
if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("cloned ->", REPO_DIR)
else:
    print("REPO_URL is empty. Either set it above, or use Option B (upload zip) below.")

**Option B — upload a zip of the repo** (run this instead of Option A if you
can't clone). Zip your local repo folder first (`zip -r jnpa-uc3-poc.zip
jnpa-uc3-poc`), then run this cell and pick the zip. You only need the
`shared/` and `ai/congestion/` trees, but uploading the whole repo is fine.

In [ ]:
# Uncomment to upload a zip instead of cloning.
# from google.colab import files
# up = files.upload()                     # pick jnpa-uc3-poc.zip
# import zipfile, os
# name = next(iter(up))
# with zipfile.ZipFile(name) as z: z.extractall(".")
# # If the zip extracts to a different top folder, set REPO_DIR accordingly:
# REPO_DIR = "jnpa-uc3-poc"
# print("extracted ->", REPO_DIR, os.listdir(REPO_DIR)[:10])

## 2. Install dependencies (CPU-only torch + the congestion service)

Installs the **CPU** torch wheel (no multi-GB CUDA), the shared library, and the
congestion package. Takes ~1–2 min on free Colab.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# CPU-only torch from the PyTorch wheel index (matches the Dockerfile).
pip("--index-url", "https://download.pytorch.org/whl/cpu", "torch>=2.0")
# Shared lib + the congestion package (editable so its config paths resolve).
pip("-e", f"{REPO_DIR}/shared")
pip("-e", f"{REPO_DIR}/ai/congestion[torch]")

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 3. Train (gap-closing knobs)

Every config value is an env var. We give the class-weight retry loop more room
(`MAX_RETRIES`) and more onset examples (`HISTORY_DAYS`) so F1 can clear 0.85.
Targets are left at the bid values (F1 0.85 / precision 0.80 / recall 0.80).

If a first run still lands just under, bump `MAX_RETRIES` to 6 and
`HISTORY_DAYS` to 28 and re-run this cell.

In [ ]:
import os, sys, importlib

# --- gap-closing overrides (all optional; remove to reproduce the baseline) ---
os.environ["CONGESTION_MAX_RETRIES"]  = "4"    # baseline 2 — more class-weight passes
os.environ["CONGESTION_HISTORY_DAYS"] = "21"   # baseline 14 — more onset examples
# Targets stay at the bid thresholds:
os.environ["CONGESTION_TARGET_F1"]        = "0.85"
os.environ["CONGESTION_TARGET_PRECISION"] = "0.80"
os.environ["CONGESTION_TARGET_RECALL"]    = "0.80"

# Put the package on the path and import the trainer.
for p in (f"{REPO_DIR}/shared", f"{REPO_DIR}/ai"):
    if p not in sys.path:
        sys.path.insert(0, p)

from congestion import train as train_mod   # noqa: E402
importlib.reload(train_mod)

summary = train_mod.train()   # writes artifacts under ai/congestion/artifacts/
print("\nF1:", round(summary["congestion_onset_f1"], 4),
      "| precision:", round(summary["precision"], 4),
      "| recall:", round(summary["recall"], 4),
      "| TARGET_MET:", summary.get("TARGET_MET"))

## 4. Inspect the artifacts that were written

In [ ]:
import json, os
ART = f"{REPO_DIR}/ai/congestion/artifacts"
print("files:", os.listdir(ART))
print("\nmetrics.json:")
print(json.dumps(json.load(open(f"{ART}/metrics.json")), indent=2)[:1200])

## 5. Download the artifacts

Unzip `congestion-artifacts.zip` into your local repo's
`ai/congestion/artifacts/`, overwriting `congestion_gnn_lstm.pt` and
`metrics.json`. Then locally re-run `scripts/poc-selftest` — check **B.1** should
now read PASS, and `tests/test_congestion.py` integration metric asserts F1 ≥ 0.85.

In [ ]:
import shutil, os
ART = f"{REPO_DIR}/ai/congestion/artifacts"
out = "congestion-artifacts"
os.makedirs(out, exist_ok=True)
for f in ("congestion_gnn_lstm.pt", "metrics.json"):
    shutil.copy(f"{ART}/{f}", f"{out}/{f}")
shutil.make_archive(out, "zip", out)

from google.colab import files
files.download(f"{out}.zip")